In [13]:
"""
What this script does:
  1. Loads cleaned data from the SQLite database created in Part A
  2. Chart 1 — Bar chart: patient count with vs without heart disease
  3. Chart 2 — Histogram: age distribution coloured by heart disease status
  4. Chart 3 — Heatmap: correlation matrix of all numerical features
 
All charts are saved as PNG files.
 
WHY EDA MATTERS:
  Before training any model, we need to visually understand the data.
  EDA helps us answer: Is the data balanced? Are there patterns?
  What features might matter most? Is there anything suspicious?
"""

'\nWhat this script does:\n  1. Loads cleaned data from the SQLite database created in Part A\n  2. Chart 1 — Bar chart: patient count with vs without heart disease\n  3. Chart 2 — Histogram: age distribution coloured by heart disease status\n  4. Chart 3 — Heatmap: correlation matrix of all numerical features\n \nAll charts are saved as PNG files.\n \nWHY EDA MATTERS:\n  Before training any model, we need to visually understand the data.\n  EDA helps us answer: Is the data balanced? Are there patterns?\n  What features might matter most? Is there anything suspicious?\n'

In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import sqlite3
import os

In [15]:
# ─────────────────────────────────────────────
# Load cleaned data from SQLite
# ─────────────────────────────────────────────
# We load directly from the database we built in Part A.
# This is good practice — one source of truth, no re-cleaning needed.
 
conn = sqlite3.connect("heart_data.db")
df = pd.read_sql_query("SELECT * FROM patients", conn)
conn.close()
 
print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns from heart_data.db")
 
# Create output folder for charts
os.makedirs("charts", exist_ok=True)
 

# ─────────────────────────────────────────────
# Shared style settings
# ─────────────────────────────────────────────
#I initially used plt.style.use('seaborn-v0_8-whitegrid') to apply a consistent white background with gridlines across all charts. However, this style name was deprecated in newer matplotlib versions, 
#so I replaced it with sns.set_theme(style='whitegrid') which achieves the same result and is the current recommended approach.
 
sns.set_theme(style="whitegrid")

 
# Our two colors: blue = no disease, red = disease
# These colors are intuitive — red signals danger/disease
COLOR_NO_DISEASE  = "#4C72B0"   # blue
COLOR_HAS_DISEASE = "#C44E52"   # red
 

Loaded 208 rows and 14 columns from heart_data.db


In [17]:
# ══════════════════════════════════════════════════════
# CHART 1: Bar Chart — Count of patients by heart disease status
# ══════════════════════════════════════════════════════
#
# PURPOSE: This is the very first chart you look at in any classification project.
# It tells you immediately whether the classes are balanced or imbalanced.
#
# WHY THIS MATTERS:
#   If one class is much larger than the other, a lazy model can just predict
#   the majority class every time and still look "accurate". That's the
#   accuracy paradox. This chart is your early warning system.
#
# WHAT WE EXPECT TO SEE:
#   127 patients without heart disease vs 81 with — a visible but not extreme gap.
#   This is a mild imbalance, not a severe one.
 
print("\nGenerating Chart 1: Bar Chart...")
 
# Count how many patients fall in each category
disease_counts = df["heart_disease"].value_counts().sort_index()
# sort_index() ensures 0 (No Disease) comes before 1 (Has Disease)
 
# Map 0/1 to readable labels
labels = ["No Heart Disease", "Has Heart Disease"]
values = [disease_counts[0], disease_counts[1]]
colors = [COLOR_NO_DISEASE, COLOR_HAS_DISEASE]
 
fig, ax = plt.subplots(figsize=(8, 6))

bars = ax.bar(labels, values, color=colors, width=0.5, edgecolor="white", linewidth=1.5)
 
# Add count labels on top of each bar
# This makes the chart self-explanatory — viewer doesn't need to read the y-axis
for bar, value in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,  # horizontal center of bar
        bar.get_height() + 2,               # just above the bar
        str(value),
        ha="center", va="bottom",
        fontsize=13, fontweight="bold"
    )
 
# Add percentage labels inside each bar
total = sum(values)
for bar, value in zip(bars, values):
    percentage = f"{value/total*100:.1f}%"
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() / 2,               # middle of bar
        percentage,
        ha="center", va="center",
        fontsize=12, color="white", fontweight="bold"
    )
 
ax.set_title("Patient Count: Heart Disease vs No Heart Disease",
             fontsize=15, fontweight="bold", pad=15)
ax.set_xlabel("Heart Disease Status", fontsize=12, labelpad=10)
ax.set_ylabel("Number of Patients",   fontsize=12, labelpad=10)
ax.set_ylim(0, max(values) + 25)  # extra space above bars for labels
ax.tick_params(labelsize=11)
 
# Add a small annotation explaining the imbalance
ax.annotate(
    f"Class ratio — No Disease: {values[0]/total*100:.0f}%  |  Disease: {values[1]/total*100:.0f}%\n"
    f"Mild class imbalance detected. Will be addressed in Part C.",
    xy=(0.5, 0.97), xycoords="axes fraction",
    ha="center", va="top", fontsize=9,
    color="gray", style="italic"
)
 
plt.tight_layout()
plt.savefig("charts/chart1_disease_distribution.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → charts/chart1_disease_distribution.png")


Generating Chart 1: Bar Chart...
  Saved → charts/chart1_disease_distribution.png


In [18]:
# ══════════════════════════════════════════════════════
# CHART 2: Histogram — Age distribution by heart disease status
# ══════════════════════════════════════════════════════
#
# PURPOSE: This chart shows whether age is a useful predictor of heart disease.
# If the two distributions overlap heavily, age alone won't be very useful.
# If they separate clearly, age is a strong signal.
#
# WHY A HISTOGRAM (not a bar chart)?
#   Age is a continuous numerical variable (29 to 76).
#   A histogram groups values into bins and shows the shape of the distribution.
#   A bar chart is for categories. A histogram is for continuous data.
#
# WHAT WE EXPECT TO SEE:
#   From SQL Query 2 we already know: avg age without disease = 52.0,
#   avg age with disease = 57.3. So the disease group should skew older.
#   The histogram makes this visible at a glance.
#
# alpha=0.6 means 60% opacity — so where the two histograms overlap,
# you can see both colors through each other. Without alpha, one would
# completely hide the other.
 
print("Generating Chart 2: Age Histogram...")
 
fig, ax = plt.subplots(figsize=(10, 6))
 
# Separate the two groups
no_disease  = df[df["heart_disease"] == 0]["age"]
has_disease = df[df["heart_disease"] == 1]["age"]
 
# bins=15 divides the age range into 15 equal-width groups
# edgecolor="white" draws thin white borders between bars — improves readability
ax.hist(no_disease,  bins=15, color=COLOR_NO_DISEASE,  alpha=0.6,
        edgecolor="white", label=f"No Heart Disease  (n={len(no_disease)})")
ax.hist(has_disease, bins=15, color=COLOR_HAS_DISEASE, alpha=0.6,
        edgecolor="white", label=f"Has Heart Disease (n={len(has_disease)})")
 
# Add vertical mean lines — they make the age shift concrete and readable
ax.axvline(no_disease.mean(),  color=COLOR_NO_DISEASE,  linestyle="--",
           linewidth=2, label=f"Mean (no disease) = {no_disease.mean():.1f}")
ax.axvline(has_disease.mean(), color=COLOR_HAS_DISEASE, linestyle="--",
           linewidth=2, label=f"Mean (disease)    = {has_disease.mean():.1f}")
 
ax.set_title("Age Distribution by Heart Disease Status",
             fontsize=15, fontweight="bold", pad=15)
ax.set_xlabel("Age (years)", fontsize=12, labelpad=10)
ax.set_ylabel("Number of Patients", fontsize=12, labelpad=10)
ax.legend(fontsize=10, framealpha=0.9)
ax.tick_params(labelsize=11)
 
# Add a text insight directly on the chart
mean_diff = has_disease.mean() - no_disease.mean()
ax.text(
    0.98, 0.95,
    f"Patients with heart disease are on average\n{mean_diff:.1f} years older",
    transform=ax.transAxes,
    ha="right", va="top", fontsize=9,
    color="dimgray", style="italic",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="lightyellow", alpha=0.8)
)
 
plt.tight_layout()
plt.savefig("charts/chart2_age_histogram.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → charts/chart2_age_histogram.png")

Generating Chart 2: Age Histogram...
  Saved → charts/chart2_age_histogram.png


In [ ]:
# Learning from above(as the overlapping was more only between 52 and 57 years of age and Doctor cannot predict from)- The less the two groups overlap, the more useful that feature is for prediction.

In [21]:
# ══════════════════════════════════════════════════════
# CHART 3: Correlation Heatmap — All numerical features
# ══════════════════════════════════════════════════════
#
# PURPOSE: This chart answers: "Which features move together?"
# A correlation of +1 means two columns increase together perfectly.
# A correlation of -1 means as one goes up, the other goes down perfectly.
# A correlation near 0 means no linear relationship.
#
# WHY THIS MATTERS FOR ML:
#   1. Feature selection — features highly correlated with 'heart_disease'
#      are likely to be powerful predictors.
#   2. Multicollinearity — if two features are highly correlated WITH EACH OTHER
#      (not with the target), having both in your model is redundant and can
#      hurt some algorithms.
#
# WHAT TO LOOK FOR:
#   - The 'heart_disease' row/column shows which features correlate with the target
#   - Warm colors (red/orange) = positive correlation
#   - Cool colors (blue) = negative correlation
#   - Deep colors = strong relationship
#
# annot=True adds the actual number in each cell — great for presentations
# fmt=".2f" formats numbers to 2 decimal places
# cmap="coolwarm" = blue (negative) → white (zero) → red (positive)

print("Generating Chart 3: Correlation Heatmap...")

import numpy as np

fig, ax = plt.subplots(figsize=(12, 9))

# Compute the full correlation matrix across all numerical columns
# select_dtypes(include='number') filters to numeric columns first
# This avoids the numeric_only argument which older pandas versions don't support
corr_matrix = df.select_dtypes(include='number').corr()

# Create a mask for the upper triangle — avoids showing the same info twice
# The heatmap is symmetric, so showing both triangles is redundant
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    mask=mask,                    # hide upper triangle
    annot=True,                   # show numbers in each cell
    fmt=".2f",                    # 2 decimal places
    cmap="coolwarm",              # blue-white-red diverging palette
    center=0,                     # white = zero correlation
    vmin=-1, vmax=1,              # fix scale to full range
    square=True,                  # each cell is a square
    linewidths=0.5,               # thin lines between cells
    linecolor="white",
    cbar_kws={"shrink": 0.8, "label": "Correlation Coefficient"},
    ax=ax
)

ax.set_title("Correlation Heatmap — All Numerical Features",
             fontsize=15, fontweight="bold", pad=20)

# Rotate axis labels for readability
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)

# Highlight key insight as annotation
ax.text(
    0.5, -0.18,
    "Key: Warm colors = positive correlation | Cool colors = negative correlation | "
    "Look at the 'heart_disease' row for predictive features",
    transform=ax.transAxes, ha="center", fontsize=8.5,
    color="dimgray", style="italic"
)

plt.tight_layout()
plt.savefig("charts/chart3_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → charts/chart3_correlation_heatmap.png")

Generating Chart 3: Correlation Heatmap...
  Saved → charts/chart3_correlation_heatmap.png


In [23]:
print("\n" + "=" * 55)
print("Part B complete. All 3 charts saved in /charts/")
print("=" * 55)
print("\nKey findings from EDA:")
print(f"  Chart 1 — Class ratio: {len(df[df.heart_disease==0])} no disease "
      f"vs {len(df[df.heart_disease==1])} disease (mild imbalance)")
print(f"  Chart 2 — Age gap: disease patients avg {df[df.heart_disease==1].age.mean():.1f} yrs "
      f"vs {df[df.heart_disease==0].age.mean():.1f} yrs (no disease)")

# FIXED: select numeric columns first, then correlate
corr_with_target = df.select_dtypes(include='number').corr()["heart_disease"].drop("heart_disease").abs().sort_values(ascending=False)
print(f"  Chart 3 — Top 3 features correlated with heart disease:")
for feat, val in corr_with_target.head(3).items():
    print(f"            {feat:30s} → {val:.2f}")


Part B complete. All 3 charts saved in /charts/

Key findings from EDA:
  Chart 1 — Class ratio: 127 no disease vs 81 disease (mild imbalance)
  Chart 2 — Age gap: disease patients avg 57.3 yrs vs 52.0 yrs (no disease)
  Chart 3 — Top 3 features correlated with heart disease:
            num_major_vessels              → 0.55
            thalassemia                    → 0.52
            st_depression                  → 0.45
